# Football Match Outcome Prediction

**Dataset:** `football_matches_2024_2025.csv`  


## Project Description

Project ini menggunakan dataset pertandingan sepak bola musim 2024/2025. Dataset berisi informasi pertandingan seperti kompetisi, tanggal pertandingan, tim tuan rumah, tim tamu, wasit, skor, dan hasil akhir pertandingan.

Tujuan project adalah membangun pipeline awal untuk **memprediksi hasil pertandingan sepak bola** berdasarkan informasi pertandingan yang tersedia di dataset.

Target prediksi yang digunakan adalah kolom:

`match_outcome`

Target tersebut memiliki 3 kategori:

- `Home Win`: tim tuan rumah menang
- `Away Win`: tim tamu menang
- `Draw`: pertandingan seri


## Case Study

**Studi kasus:**  
Sebuah sistem analisis sepak bola ingin membantu memprediksi kemungkinan hasil pertandingan berdasarkan data historis pertandingan. Prediksi ini dapat digunakan sebagai dasar analisis performa tim, evaluasi pertandingan, atau bahan eksperimen model klasifikasi.

Pada tahap ini, kita menyiapkan data agar siap digunakan untuk pelatihan model. Tahapan preprocessing meliputi:

1. Load dataset menggunakan pandas.
2. Menampilkan informasi dataset.
3. Mengecek missing value.
4. Memilih fitur dan target.
5. Melakukan encoding data kategori.
6. Membagi data menjadi training dan testing.
7. Melakukan normalisasi data.

**Catatan penting:**  
Kolom hasil pertandingan seperti `fulltime_home`, `fulltime_away`, `halftime_home`, `halftime_away`, `goal_difference`, `total_goals`, `home_points`, dan `away_points` tidak digunakan sebagai fitur utama karena kolom tersebut berisi informasi setelah pertandingan selesai. Jika kolom tersebut digunakan untuk memprediksi `match_outcome`, model bisa terlalu mudah menebak hasil karena terjadi *data leakage*.


# Langkah 1 — Preprocessing Data

## 1. Import Library

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option("display.max_columns", None)


## 2. Load Dataset Menggunakan Pandas

In [ ]:
# Nama file dataset
DATASET_NAME = "football_matches_2024_2025.csv"

# Cek apakah file ada di folder kerja saat ini
if os.path.exists(DATASET_NAME):
    data_path = DATASET_NAME
else:
    # Fallback khusus Google Colab: upload file secara manual
    try:
        from google.colab import files
        print("File dataset belum ditemukan. Silakan upload file CSV.")
        uploaded = files.upload()
        data_path = list(uploaded.keys())[0]
    except ModuleNotFoundError:
        raise FileNotFoundError(
            f"File {DATASET_NAME} belum ditemukan. "
            "Pastikan file CSV berada di folder yang sama dengan notebook."
        )

df = pd.read_csv(data_path)

print("Dataset berhasil dimuat.")
print("Jumlah baris dan kolom:", df.shape)

df.head()


## 3. Menampilkan Informasi Dataset

In [ ]:
# Melihat informasi umum dataset
df.info()


In [ ]:
# Melihat nama kolom dan tipe data
info_dataset = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Unique Values": df.nunique().values
})

info_dataset


In [ ]:
# Statistik deskriptif untuk data numerik
df.describe()


In [ ]:
# Statistik deskriptif untuk data kategori
df.describe(include="object")


## 4. Mengecek Missing Value

In [ ]:
missing_values = pd.DataFrame({
    "Missing Value": df.isnull().sum(),
    "Percentage (%)": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values


In [ ]:
# Menampilkan total missing value
total_missing = df.isnull().sum().sum()
print("Total missing value:", total_missing)


## 5. Memilih Fitur dan Target

Target yang dipilih adalah `match_outcome` karena project ini berupa klasifikasi hasil pertandingan.

Fitur yang digunakan adalah fitur yang masih relevan sebelum hasil akhir pertandingan diketahui, seperti kompetisi, matchday, stage, tim, wasit, dan informasi waktu pertandingan.

Kolom skor akhir dan poin tidak digunakan karena dapat menyebabkan *data leakage*.


In [ ]:
# Mengubah kolom date_utc menjadi datetime
df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce", utc=True)

# Membuat fitur tambahan dari tanggal pertandingan
df["match_month"] = df["date_utc"].dt.month
df["match_dayofweek"] = df["date_utc"].dt.dayofweek

# Target
target_col = "match_outcome"

# Kolom kategori yang akan di-encoding
categorical_features = [
    "competition_code",
    "competition_name",
    "stage",
    "status",
    "referee",
    "home_team",
    "away_team"
]

# Kolom numerik yang akan dinormalisasi
numeric_features = [
    "matchday",
    "home_team_id",
    "away_team_id",
    "referee_id",
    "match_month",
    "match_dayofweek"
]

# Fitur dan target
feature_cols = categorical_features + numeric_features

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Jumlah fitur:", len(feature_cols))
print("Fitur yang digunakan:")
print(feature_cols)

print("\nTarget yang digunakan:", target_col)
print("\nDistribusi target:")
print(y.value_counts())


## 6. Encoding Data Kategori

Data kategori seperti nama tim, nama kompetisi, stage, status, dan wasit perlu diubah menjadi bentuk numerik agar dapat diproses oleh model Machine Learning.

Pada notebook ini:

- Fitur kategori diubah menggunakan **One-Hot Encoding**.
- Target `match_outcome` diubah menggunakan **Label Encoding**.


In [ ]:
# Encoding target menggunakan LabelEncoder
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Mapping label target:")
for class_name, class_code in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"{class_name} -> {class_code}")

print("\nContoh hasil encoding target:")
pd.DataFrame({
    "match_outcome": y.head(10),
    "encoded_target": y_encoded[:10]
})


In [ ]:
# Membuat OneHotEncoder
# Catatan: sparse_output digunakan di scikit-learn versi baru.
# Jika versi lama, gunakan parameter sparse.
try:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

# Preprocessor untuk fitur:
# - OneHotEncoder untuk data kategori
# - StandardScaler untuk data numerik
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", onehot_encoder, categorical_features),
        ("numeric", StandardScaler(), numeric_features)
    ],
    remainder="drop"
)

preprocessor


## 7. Membagi Data Menjadi Training dan Testing

Data dibagi menjadi:

- 80% data training
- 20% data testing

Parameter `stratify=y_encoded` digunakan agar proporsi kelas target pada data training dan testing tetap seimbang.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test:", X_test.shape)
print("Ukuran y_train:", y_train.shape)
print("Ukuran y_test:", y_test.shape)


In [ ]:
# Mengecek distribusi target setelah split
train_distribution = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_distribution = pd.Series(y_test).value_counts(normalize=True).sort_index()

distribution_df = pd.DataFrame({
    "Class": label_encoder.classes_,
    "Train Distribution": train_distribution.values.round(3),
    "Test Distribution": test_distribution.values.round(3)
})

distribution_df


## 8. Normalisasi Data

Normalisasi dilakukan menggunakan `StandardScaler` untuk fitur numerik.  
Agar preprocessing aman, proses `fit` hanya dilakukan pada data training, lalu data testing hanya ditransform menggunakan scaler/encoder yang sudah dipelajari dari data training.

Pada kode di bawah, encoding dan normalisasi dilakukan sekaligus melalui `ColumnTransformer`.


In [ ]:
# Fit hanya pada data training, lalu transform train dan test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Shape data training setelah preprocessing:", X_train_processed.shape)
print("Shape data testing setelah preprocessing:", X_test_processed.shape)


In [ ]:
# Mengambil nama kolom hasil one-hot encoding
encoded_cat_cols = preprocessor.named_transformers_["categorical"].get_feature_names_out(categorical_features)
processed_feature_names = list(encoded_cat_cols) + numeric_features

# Mengubah hasil preprocessing ke DataFrame agar mudah dilihat
X_train_processed_df = pd.DataFrame(X_train_processed, columns=processed_feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=processed_feature_names)

X_train_processed_df.head()


## 9. Hasil Akhir Preprocessing

Data sudah siap digunakan untuk tahap modeling, misalnya menggunakan Neural Network sederhana.

Output yang dihasilkan:

- `X_train_processed`
- `X_test_processed`
- `y_train`
- `y_test`
- `label_encoder`
- `preprocessor`


In [ ]:
print("Ringkasan hasil akhir:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)
print("y_train          :", y_train.shape)
print("y_test           :", y_test.shape)

print("\nClass target:")
print(label_encoder.classes_)


In [ ]:
# Contoh data akhir yang siap masuk ke model
final_train_preview = X_train_processed_df.copy()
final_train_preview["target"] = y_train

final_train_preview.head()


## 10. Kesimpulan Preprocessing

Berdasarkan tahapan preprocessing yang sudah dilakukan:

1. Dataset berhasil dimuat menggunakan pandas.
2. Informasi dataset berhasil ditampilkan, termasuk jumlah baris, kolom, tipe data, dan nilai unik.
3. Missing value berhasil dicek.
4. Fitur dan target berhasil dipilih.
5. Data kategori berhasil disiapkan dengan One-Hot Encoding.
6. Target `match_outcome` berhasil diubah menjadi label numerik.
7. Dataset berhasil dibagi menjadi data training dan testing.
8. Fitur numerik berhasil dinormalisasi menggunakan StandardScaler.
9. Data akhir sudah siap digunakan untuk proses training model Machine Learning atau Neural Network.
